In [2]:
from google.colab import drive
import shutil
shutil.copy("/content/drive/MyDrive/Dataset/skin_disease_resnet50v2.keras", "/content/skin_disease_resnet50v2.keras")
print("Model restored successfully!")

Model restored successfully!


In [3]:
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio

In [4]:
!pip install opencv-python-headless

In [5]:
# Cell 1: Remove old cached app module
import sys
if 'app' in sys.modules:
    del sys.modules['app']

import os
if os.path.exists('app.py'):
    os.remove('app.py')
print("Cleared old app.py and module cache")

Cleared old app.py and module cache


In [6]:
%%writefile app.py
import io
import math
import base64
import numpy as np
import cv2
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image
import tensorflow as tf

app = FastAPI(title="Skin Disease Prediction API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

MODEL_PATH = "skin_disease_resnet50v2.keras"
IMG_SIZE = (260, 260)
TTA_STEPS = 8
CLASS_NAMES = [
    "Acne",
    "Blackheads",
    "Dark-Spots",
    "Dry-Skin",
    "Englarged-Pores",   # ← typo with 'l' — keep as-is to match folder
    "Eyebags",
    "Normal Skin",       # ← SPACE not dash — this was your bug!
    "Not-Skin",
    "Oily-Skin",
    "Skin-Redness",
    "Whiteheads",
    "Wrinkles"
]

class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        n_cls  = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true = y_true * (1.0 - self.label_smoothing) + self.label_smoothing / n_cls
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce     = -y_true * tf.math.log(y_pred)
        weight = tf.pow(1.0 - y_pred, self.gamma)
        focal  = weight * ce
        return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"gamma": self.gamma, "label_smoothing": self.label_smoothing})
        return cfg
model = None

@app.on_event("startup")
def load_model():
    global model
    try:
        model = tf.keras.models.load_model(MODEL_PATH, custom_objects={"FocalLoss": FocalLoss})
        print("Successfully loaded Keras model inside Colab.")
    except Exception as e:
        print(f"Error loading model: {e}")

resnet_preprocess = tf.keras.applications.resnet_v2.preprocess_input

def random_rotate_tf(image, max_deg=25.0):
    angle = tf.random.uniform([], -max_deg, max_deg) * (math.pi / 180.0)
    ca, sa = tf.math.cos(angle), tf.math.sin(angle)
    h, w = tf.cast(tf.shape(image)[0], tf.float32), tf.cast(tf.shape(image)[1], tf.float32)
    cx, cy = w / 2.0, h / 2.0
    t = [ca, -sa, cx - cx*ca + cy*sa, sa, ca, cy - cx*sa - cy*ca, 0.0, 0.0]
    t = tf.reshape(tf.stack(t), [1, 8])
    return tf.raw_ops.ImageProjectiveTransformV3(
        images=tf.expand_dims(image, 0), transforms=t,
        output_shape=tf.shape(image)[:2],
        interpolation="BILINEAR", fill_mode="REFLECT", fill_value=0.0)[0]

def random_zoom_tf(image, lo=0.85, hi=1.15):
    factor = tf.random.uniform([], lo, hi)
    h, w = tf.shape(image)[0], tf.shape(image)[1]
    new_h = tf.maximum(tf.cast(tf.cast(h, tf.float32) / factor, tf.int32), 1)
    new_w = tf.maximum(tf.cast(tf.cast(w, tf.float32) / factor, tf.int32), 1)
    image = tf.image.resize_with_crop_or_pad(image, new_h, new_w)
    return tf.image.resize_with_crop_or_pad(image, h, w)

def augment_image(image_tensor):
    image = tf.image.random_flip_left_right(image_tensor)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, 0.30)
    image = tf.image.random_contrast(image, 0.70, 1.30)
    image = tf.image.random_saturation(image, 0.60, 1.40)
    image = tf.image.random_hue(image, 0.06)
    image = random_rotate_tf(image, max_deg=25.0)
    image = random_zoom_tf(image, lo=0.85, hi=1.15)
    return tf.clip_by_value(image, 0.0, 255.0)

def has_enough_skin(image_bytes: bytes, min_skin_ratio: float = 0.10) -> bool:
    """Check if image contains enough skin pixels to be a valid skin condition image."""
    # Decode bytes directly instead of reading from file path
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    if img is None:
        return False

    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

    lower = np.array([0, 133, 77], dtype=np.uint8)
    upper = np.array([255, 173, 127], dtype=np.uint8)

    skin_mask = cv2.inRange(img_ycrcb, lower, upper)
    skin_ratio = np.sum(skin_mask > 0) / skin_mask.size

    print(f"Skin pixel ratio: {skin_ratio:.2%}")
    return skin_ratio >= min_skin_ratio

def run_prediction(contents: bytes, use_tta: bool):
    if model is None:
        raise HTTPException(status_code=500, detail="Model not loaded on server.")

    if not has_enough_skin(contents):
        return {
        "success": False,
        "detail":"Image does not appear to contain sufficient skin. Please upload a clear photo of the affected skin area."
        }

    img_pil = Image.open(io.BytesIO(contents)).convert("RGB").resize(IMG_SIZE)
    img_f = tf.cast(np.array(img_pil), tf.float32)
    orig = resnet_preprocess(img_f)

    if use_tta:
        batch = [orig] + [resnet_preprocess(augment_image(img_f)) for _ in range(TTA_STEPS - 1)]
        batch = tf.stack(batch)
        aug_probs = model.predict(batch, verbose=0)
    else:
        aug_probs = model.predict(tf.expand_dims(orig, 0), verbose=0)

    final_probs = np.mean(aug_probs, axis=0)
    top5_indices = final_probs.argsort()[-5:][::-1]

    predictions = [
        {"class_name": CLASS_NAMES[idx] if idx < len(CLASS_NAMES) else f"Class {idx}", "confidence": float(final_probs[idx])}
        for idx in top5_indices
    ]

    return {
        "success": True,
        "prediction": predictions[0]["class_name"],
        "confidence": predictions[0]["confidence"],
        "top_5": predictions
    }

# ── Multipart form-data endpoint (used by the HTML/Colab camera frontend) ──
@app.post("/predict")
async def predict(file: UploadFile = File(...), use_tta: bool = True):
    if not file.content_type or not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="Uploaded file must be an image.")
    try:
        contents = await file.read()
        return run_prediction(contents, use_tta)
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

# ── JSON + base64 endpoint (used by the React Native app) ──
class ImagePayload(BaseModel):
    file: str
    use_tta: bool = True

@app.post("/predict_base64")
async def predict_base64(payload: ImagePayload):
    try:
        try:
            contents = base64.b64decode(payload.file)
        except Exception:
            raise HTTPException(status_code=400, detail="Invalid base64 image data.")
        return run_prediction(contents, payload.use_tta)
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": model is not None}

Writing app.py


In [7]:
# Cell 2: Verify the file was written correctly
with open('app.py', 'r') as f:
    content = f.read()

# Check CLASS_NAMES is correct
assert '"Normal Skin"' in content, "ERROR: Normal Skin missing!"
assert '"Not-Skin"' in content, "ERROR: Not-Skin missing!"
assert 'return tf.reduce_mean(y_pred)' not in content, "ERROR: broken FocalLoss still there!"
print("✅ app.py looks correct")

✅ app.py looks correct


In [ ]:
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
import nest_asyncio, uvicorn
from pyngrok import ngrok

ngrok.kill()  # close any leftover tunnels from previous runs
nest_asyncio.apply()

from app import app  # imports the FastAPI app defined in app.py

PORT = 8000
public_url = ngrok.connect(PORT).public_url
print(f"Public API URL: {public_url}")
print(f"Predict route : {public_url}/predict?use_tta=false")
print(f"Docs          : {public_url}/docs")

config = uvicorn.Config(app, host="0.0.0.0", port=PORT, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [3022]
INFO:     Waiting for application startup.


Public API URL: https://greeter-darling-prowess.ngrok-free.dev
Predict route : https://greeter-darling-prowess.ngrok-free.dev/predict?use_tta=false
Docs          : https://greeter-darling-prowess.ngrok-free.dev/docs


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 169 variables whereas the saved optimizer has 173 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Successfully loaded Keras model inside Colab.
INFO:     2407:d000:11:425f:d8ec:27ab:82bc:9eff:0 - "OPTIONS /predict_base64 HTTP/1.1" 200 OK
Skin pixel ratio: 4.98%
INFO:     2407:d000:11:425f:d8ec:27ab:82bc:9eff:0 - "POST /predict_base64 HTTP/1.1" 200 OK
Skin pixel ratio: 13.13%
INFO:     2407:d000:11:425f:d8ec:27ab:82bc:9eff:0 - "POST /predict_base64 HTTP/1.1" 200 OK
Skin pixel ratio: 10.90%
INFO:     2407:d000:11:425f:d8ec:27ab:82bc:9eff:0 - "POST /predict_base64 HTTP/1.1" 200 OK
